# Плотный локализатор городищ на спутнике (SpOr) без лидара: DINOv2-L с регистрами (bot-5)
**Только честный замер. Ничего не идёт в сдачу до письма организаторов.** Запуск — bot 9.

Что делает: (1) `scripts/gor_dense/dense_feats.py` — вся сцена SpOr на фиксированном шаге (GSDS м/пкс), тайлы 518 px с перекрытием,
плотные патч-токены замороженного `facebook/dinov2-with-registers-large` (Apache-2.0), две шкалы склеены; (2) `scripts/gor_dense/dense_probe.py` —
линейная проба по токенам с LOFO/honest-10/seed0-4 (те же фолды и метки, что на боксе: `kaggle/gor_dense_inputs.json`), PU-игнор, трудные негативы,
сглаживание, пики → `peaks_*.json` (OOF-пики отложенных участков + внутренний OOF для обучающих). Офиц. метрика и П1–П5 — только на боксе bot-5.

Inputs: репо `expds-koz3-bot5` (ветка `bot5-gor-dense-loc`), датасет `train-expds-3` (нужны `*/*SpOr*/*.tif*` и `*/UTM.json`).
Accelerator: GPU T4 (×1 достаточно), Internet ON (веса с HF).

In [ ]:
REPO='/kaggle/input/expds-koz3-bot5'   # поправить на фактический путь
!pip install -q -r $REPO/kaggle/requirements_gor.txt 2>&1 | tail -1
import torch, transformers, scipy; print(torch.__version__, transformers.__version__, scipy.__version__, torch.cuda.device_count())
assert transformers.__version__ == '5.21.0', transformers.__version__
import os, shutil
os.environ.update(DATA='/kaggle/input/train-expds-3', OUT='/kaggle/working/gd', DOUT='/kaggle/working/gd', MODEL='facebook/dinov2-with-registers-large', REVISION='e4c89a4e05589de9b3e188688a303d0f3c04d0f3',
                  DEVICE='cuda', THREADS='4', REPO=REPO)
os.makedirs('/kaggle/working/gd', exist_ok=True)
shutil.copy(f'{REPO}/kaggle/gor_dense_inputs.json', '/kaggle/working/gd/gor_dense_inputs.json')
!ls $DATA | head -3

## Шаг 1 — smoke (обязателен первым): один участок, обе шкалы

In [ ]:
!ONLY_REGS=185_Изборск GSDS=1,2 TAG=_smoke python $REPO/scripts/gor_dense/dense_feats.py 2>&1 | tail -4
import numpy as np, json; z = np.load('/kaggle/working/gd/dense_smoke.npz'); print(z['F'].shape, z['F'].dtype, np.isfinite(z['F'].astype('f4')).all())
assert z['F'].shape[1] == 2048, 'ожидается 1024 x 2 шкалы'

## Шаг 2 — признаки, основная шкала 1 и 2 м/пкс (`_L12`) и контрольная 2 и 4 м/пкс (`_L24`, как CPU-прототип на DINOv2-S)
Ориентир по объёму (бокс, DINOv2-S, 2/4 м/пкс): 175 файлов, 227 182 токена, 580 тайлов, 552 с на 2 потоках CPU (`analysis/gor_dense/dense_feats.log`).
При 1/2 м/пкс токенов ≈ в 4 раза больше (≈0,9 млн × 2048 × fp16 ≈ 3,7 ГБ на диске и в RAM) — помещается в Kaggle (30 ГБ RAM, 20 ГБ диска).

In [ ]:
!GSDS=2,4 TAG=_L24 python $REPO/scripts/gor_dense/dense_feats.py 2>&1 | tail -2
!GSDS=1,2 TAG=_L12 python $REPO/scripts/gor_dense/dense_feats.py 2>&1 | tail -2
!ls -la /kaggle/working/gd; df -h /kaggle/working | tail -1

## Шаг 3 — проба по фолдам (CPU, numpy) → peaks_*.json. Сначала `_L24` (быстрее), затем `_L12`.

In [ ]:
!OMP_NUM_THREADS=4 TAG=_L24 python $REPO/scripts/gor_dense/dense_probe.py 2>&1 | grep -v '^$' | tail -12
!OMP_NUM_THREADS=4 TAG=_L12 python $REPO/scripts/gor_dense/dense_probe.py 2>&1 | grep -v '^$' | tail -12

## Шаг 4 — выгрузка (маленький архив: только пики, логи и описание файлов; dense*.npz НЕ нужен)

In [ ]:
!cd /kaggle/working/gd && tar czf /kaggle/working/gor_dense_L.tgz peaks_L24.json peaks_L12.json files_L24.json files_L12.json && ls -la /kaggle/working/gor_dense_L.tgz
!rm -f /kaggle/working/gd/dense_L12.npz /kaggle/working/gd/dense_smoke.npz   # не выгружать гигабайты

Вернуть `gor_dense_L.tgz` bot-5. На боксе: `tar xzf gor_dense_L.tgz -C analysis/gor_dense` →
`DTAG=_L12 python scripts/gor_dense/gor_dense_eval.py` (и `_L24`) → `analysis/gor_dense/gor_dense_eval_L12.json`, решение по П1–П5.